# 05 — MegaDetector and Lighting Analysis

Final diagnostic notebook for the current workflow.

Focus:
- evaluate the fine-tuned MobileNetV2 model on the validation set;
- estimate performance by automatic lighting groups;
- optionally merge MegaDetector outputs to test whether localization/animal detection explains errors;
- optionally run a separate higher-resolution experiment without combining it with detector results.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import sys
import json
import random
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT.resolve()) not in sys.path:
    sys.path.append(str(PROJECT_ROOT.resolve()))

from src.download_data import READABLE_LABELS

DATA_DIR = PROJECT_ROOT / "data"
METADATA_DIR = DATA_DIR / "metadata"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MD_DIR = REPORTS_DIR / "megadetector"

TRAIN_CSV = METADATA_DIR / "train_1000.csv"
VAL_CSV = METADATA_DIR / "val_1000.csv"
TEST_CSV = METADATA_DIR / "test_1000.csv"

BEST_MODEL_PATH = MODELS_DIR / "mobilenetv2_finetuned_1000_per_class.keras"
VAL_PREDICTIONS_PATH = REPORTS_DIR / "05_val_predictions_finetuned_1000_per_class.csv"
LIGHTING_PATH = REPORTS_DIR / "05_val_lighting_labels.csv"
MD_JSON_PATH = MD_DIR / "megadetector_val_results.json"
MD_SUMMARY_PATH = REPORTS_DIR / "05_megadetector_val_summary.csv"

IMG_SIZE = (224, 224)
BATCH_SIZE = 64

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Best model path: {BEST_MODEL_PATH}")
print(f"Validation CSV: {VAL_CSV}")

## 2. Load validation metadata

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

for df in [train_df, val_df, test_df]:
    if "readable_label" not in df.columns:
        df["readable_label"] = df["label"].map(READABLE_LABELS)

class_names = sorted(train_df["readable_label"].dropna().unique())
label_to_idx = {label: idx for idx, label in enumerate(class_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
num_classes = len(class_names)

for df in [train_df, val_df, test_df]:
    df["label_idx"] = df["readable_label"].map(label_to_idx)


def resolve_image_path(row):
    if "cropped_path_rel" in row and pd.notna(row["cropped_path_rel"]):
        return PROJECT_ROOT / row["cropped_path_rel"]
    if "cropped_path" in row and pd.notna(row["cropped_path"]):
        path = Path(row["cropped_path"])
        return path if path.is_absolute() else PROJECT_ROOT / path
    raise ValueError("No cropped image path found.")


for df in [train_df, val_df, test_df]:
    df["image_path"] = df.apply(resolve_image_path, axis=1)

split_counts = pd.DataFrame({
    "train": train_df["readable_label"].value_counts().sort_index(),
    "val": val_df["readable_label"].value_counts().sort_index(),
    "test": test_df["readable_label"].value_counts().sort_index(),
}).fillna(0).astype(int)

print(f"Classes: {num_classes}")
display(split_counts)

## 3. Evaluate the fine-tuned MobileNetV2 model

This section records the current best validation performance before adding detector or lighting analysis.

In [ ]:
def load_image_array(image_path, image_size=IMG_SIZE):
    image = Image.open(image_path).convert("RGB")
    image = image.resize(image_size)
    return np.asarray(image, dtype=np.float32) / 255.0


class ImageSequence(keras.utils.Sequence):
    def __init__(self, df, batch_size=32, image_size=IMG_SIZE):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.image_size = image_size

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]
        images = [load_image_array(row["image_path"], self.image_size) for _, row in batch_df.iterrows()]
        labels = batch_df["label_idx"].astype("int32").values
        return np.stack(images), labels


val_seq = ImageSequence(val_df, batch_size=BATCH_SIZE, image_size=IMG_SIZE)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing model: {BEST_MODEL_PATH}")

model = keras.models.load_model(BEST_MODEL_PATH, safe_mode=False)
print(f"Loaded model: {BEST_MODEL_PATH.name}")

In [ ]:
def get_predictions(model, dataset):
    y_true = []
    y_prob = []

    for batch_idx in range(len(dataset)):
        images, labels = dataset[batch_idx]
        probs = model(images, training=False).numpy()
        y_true.extend(labels)
        y_prob.extend(probs)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = np.argmax(y_prob, axis=1)
    return y_true, y_pred, y_prob


def build_results_df(df, y_true, y_pred, y_prob):
    results_df = df.copy().reset_index(drop=True)
    top2_idx = np.argsort(y_prob, axis=1)[:, -2]

    results_df["image_path"] = results_df["image_path"].astype(str)
    results_df["true_idx"] = y_true
    results_df["pred_idx"] = y_pred
    results_df["true_label"] = results_df["true_idx"].map(idx_to_label)
    results_df["pred_label"] = results_df["pred_idx"].map(idx_to_label)
    results_df["confidence"] = np.max(y_prob, axis=1)
    results_df["correct"] = results_df["true_idx"] == results_df["pred_idx"]
    results_df["top2_idx"] = top2_idx
    results_df["top2_label"] = results_df["top2_idx"].map(idx_to_label)
    results_df["top2_confidence"] = y_prob[np.arange(len(y_prob)), top2_idx]
    results_df["prediction_margin"] = results_df["confidence"] - results_df["top2_confidence"]

    for idx, class_name in idx_to_label.items():
        results_df[f"prob_{class_name}"] = y_prob[:, idx]

    return results_df


def arrays_from_results_df(results_df):
    prob_columns = [f"prob_{class_name}" for class_name in class_names]
    y_true = results_df["true_idx"].to_numpy(dtype=np.int32)
    y_pred = results_df["pred_idx"].to_numpy(dtype=np.int32)
    y_prob = results_df[prob_columns].to_numpy(dtype=np.float32)
    return y_true, y_pred, y_prob


use_cache = VAL_PREDICTIONS_PATH.exists()
if use_cache:
    val_results_df = pd.read_csv(VAL_PREDICTIONS_PATH)
    use_cache = len(val_results_df) == len(val_df) and set(val_results_df["true_label"].unique()) == set(class_names)

if use_cache:
    y_val_true, y_val_pred, y_val_prob = arrays_from_results_df(val_results_df)
    print(f"Loaded cached predictions: {VAL_PREDICTIONS_PATH}")
else:
    y_val_true, y_val_pred, y_val_prob = get_predictions(model, val_seq)
    val_results_df = build_results_df(val_df, y_val_true, y_val_pred, y_val_prob)
    val_results_df.to_csv(VAL_PREDICTIONS_PATH, index=False)
    print(f"Saved predictions: {VAL_PREDICTIONS_PATH}")

val_accuracy = accuracy_score(y_val_true, y_val_pred)
val_macro_f1 = f1_score(y_val_true, y_val_pred, average="macro")

print(f"Fine-tuned MobileNetV2 validation accuracy: {val_accuracy:.4f}")
print(f"Fine-tuned MobileNetV2 validation macro-F1:  {val_macro_f1:.4f}")

display(val_results_df.head())

In [ ]:
report_df = pd.DataFrame(
    classification_report(
        y_val_true,
        y_val_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).T

display(report_df)

cm = confusion_matrix(y_val_true, y_val_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax,
    xticks_rotation=45,
    cmap="Blues",
    colorbar=False,
)
ax.set_title("Fine-tuned MobileNetV2 validation confusion matrix")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "05_finetuned_mobilenetv2_val_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Automatic lighting labels

Lighting labels are heuristic. They are used for diagnosis, not as ground-truth annotations.

In [ ]:
def compute_lighting_features(image_path, resize_to=(256, 256)):
    image = Image.open(image_path).convert("RGB")
    image = image.resize(resize_to)
    arr = np.asarray(image, dtype=np.float32) / 255.0

    brightness = arr.mean(axis=2)
    max_channel = arr.max(axis=2)
    min_channel = arr.min(axis=2)
    saturation = (max_channel - min_channel) / (max_channel + 1e-6)

    return {
        "mean_brightness": float(brightness.mean()),
        "median_brightness": float(np.median(brightness)),
        "std_brightness": float(brightness.std()),
        "dark_fraction": float((brightness < 0.20).mean()),
        "very_dark_fraction": float((brightness < 0.10).mean()),
        "bright_fraction": float((brightness > 0.80).mean()),
        "mean_saturation": float(saturation.mean()),
        "channel_std_mean": float(arr.std(axis=2).mean()),
    }


def assign_lighting_label(row):
    if row["mean_brightness"] < 0.22 or row["dark_fraction"] > 0.65:
        return "night_or_low_light"
    if row["channel_std_mean"] < 0.035 and row["mean_saturation"] < 0.15:
        return "grayscale_or_ir"
    if row["mean_brightness"] < 0.35 or row["dark_fraction"] > 0.40:
        return "low_light"
    return "day_or_normal_light"


if LIGHTING_PATH.exists():
    lighting_df = pd.read_csv(LIGHTING_PATH)
    print(f"Loaded lighting labels: {LIGHTING_PATH}")
else:
    lighting_rows = []
    for _, row in val_df.reset_index(drop=True).iterrows():
        features = compute_lighting_features(row["image_path"])
        features["image_id"] = row["image_id"]
        features["image_path"] = str(row["image_path"])
        lighting_rows.append(features)

    lighting_df = pd.DataFrame(lighting_rows)
    lighting_df["lighting_label"] = lighting_df.apply(assign_lighting_label, axis=1)
    lighting_df.to_csv(LIGHTING_PATH, index=False)
    print(f"Saved lighting labels: {LIGHTING_PATH}")

display(lighting_df["lighting_label"].value_counts().reset_index(name="count"))
display(lighting_df.head())

## 5. Performance by lighting condition

In [ ]:
analysis_df = val_results_df.merge(
    lighting_df[[
        "image_id",
        "lighting_label",
        "mean_brightness",
        "dark_fraction",
        "mean_saturation",
        "channel_std_mean",
    ]],
    on="image_id",
    how="left",
)


def macro_f1_for_group(group):
    labels_present = sorted(group["true_idx"].unique())
    return f1_score(
        group["true_idx"],
        group["pred_idx"],
        average="macro",
        labels=labels_present,
        zero_division=0,
    )

lighting_summary = (
    analysis_df.groupby("lighting_label")
    .apply(lambda g: pd.Series({
        "count": len(g),
        "accuracy": accuracy_score(g["true_idx"], g["pred_idx"]),
        "macro_f1_present_classes": macro_f1_for_group(g),
        "mean_confidence": g["confidence"].mean(),
        "error_rate": 1.0 - g["correct"].mean(),
    }))
    .reset_index()
    .sort_values("accuracy")
)

lighting_summary.to_csv(REPORTS_DIR / "05_performance_by_lighting.csv", index=False)
display(lighting_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(lighting_summary["lighting_label"], lighting_summary["accuracy"])
ax.set_ylim(0, 1)
ax.set_xlabel("Lighting label")
ax.set_ylabel("Validation accuracy")
ax.set_title("Validation accuracy by automatic lighting label")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "05_accuracy_by_lighting.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
error_df = analysis_df[~analysis_df["correct"]].copy()

lighting_error_predictions = pd.crosstab(
    error_df["lighting_label"],
    error_df["pred_label"],
    normalize="index",
).fillna(0)

display(lighting_error_predictions)

raccoon_fp_df = analysis_df[
    (analysis_df["pred_label"] == "raccoon")
    & (analysis_df["true_label"] != "raccoon")
].copy()

raccoon_fp_by_lighting = (
    raccoon_fp_df.groupby("lighting_label")
    .size()
    .reset_index(name="raccoon_false_positives")
    .sort_values("raccoon_false_positives", ascending=False)
)

display(raccoon_fp_by_lighting)

In [ ]:
analysis_df["true_binary"] = np.where(analysis_df["true_label"] == "empty", "empty", "non_empty")
analysis_df["pred_binary"] = np.where(analysis_df["pred_label"] == "empty", "empty", "non_empty")

binary_by_lighting = (
    analysis_df.groupby("lighting_label")
    .apply(lambda g: pd.Series({
        "count": len(g),
        "binary_accuracy": accuracy_score(g["true_binary"], g["pred_binary"]),
        "animal_predicted_empty": int(((g["true_binary"] == "non_empty") & (g["pred_binary"] == "empty")).sum()),
        "empty_predicted_animal": int(((g["true_binary"] == "empty") & (g["pred_binary"] == "non_empty")).sum()),
    }))
    .reset_index()
)

display(binary_by_lighting)

## 6. Visual checks for lighting-related errors

In [ ]:
def show_examples(df, title, n=12, ncols=4, sort_by="confidence", ascending=False, save_path=None):
    examples = df.copy()
    if sort_by in examples.columns:
        examples = examples.sort_values(sort_by, ascending=ascending)
    examples = examples.head(n)

    if examples.empty:
        print("No examples to show.")
        return None

    nrows = int(np.ceil(len(examples) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.0 * ncols, 4.6 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, examples.iterrows()):
        image = plt.imread(row["image_path"])
        ax.imshow(image)
        ax.set_title(
            f"T: {row['true_label']} | P: {row['pred_label']}\n"
            f"Conf: {row['confidence']:.2f} | {row['lighting_label']}",
            fontsize=9,
        )
        ax.axis("off")

    for ax in axes[len(examples):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fig


night_errors = analysis_df[
    (~analysis_df["correct"])
    & (analysis_df["lighting_label"].isin(["night_or_low_light", "low_light", "grayscale_or_ir"]))
]

show_examples(
    night_errors,
    title="High-confidence errors in low-light groups",
    n=12,
    ncols=4,
    save_path=FIGURES_DIR / "05_low_light_high_confidence_errors.png",
)

In [ ]:
empty_to_animal_low_light = analysis_df[
    (analysis_df["true_label"] == "empty")
    & (analysis_df["pred_label"] != "empty")
    & (analysis_df["lighting_label"].isin(["night_or_low_light", "low_light", "grayscale_or_ir"]))
]

show_examples(
    empty_to_animal_low_light,
    title="Low-light empty images predicted as animal",
    n=12,
    ncols=4,
    save_path=FIGURES_DIR / "05_low_light_empty_predicted_as_animal.png",
)

## 7. MegaDetector setup

Run MegaDetector externally if no JSON exists yet. This notebook expects one JSON file with detections for the validation images.

In [ ]:
VAL_IMAGE_LIST_PATH = MD_DIR / "val_image_paths.txt"
val_df["image_path"].astype(str).to_csv(VAL_IMAGE_LIST_PATH, index=False, header=False)

print(f"Validation image list written to: {VAL_IMAGE_LIST_PATH}")
print(f"Expected MegaDetector output JSON: {MD_JSON_PATH}")

print("Suggested command after installing MegaDetector:")
print()
print(
    "python -m megadetector.detection.run_detector_batch "
    "<PATH_TO_MEGDETECTOR_MODEL.pt> "
    f"{VAL_IMAGE_LIST_PATH} "
    f"{MD_JSON_PATH} "
    "--recursive"
)

## 8. Parse MegaDetector output

This section runs only after `megadetector_val_results.json` exists.

In [ ]:
def detection_area_ratio(det):
    bbox = det.get("bbox", None)
    if bbox is None or len(bbox) != 4:
        return 0.0
    _, _, w, h = bbox
    return float(w) * float(h)


def parse_megadetector_json(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    rows = []
    for image_result in data.get("images", []):
        image_path = image_result.get("file") or image_result.get("filename")
        detections = image_result.get("detections", []) or []
        failure = image_result.get("failure", None)

        animal_detections = [
            det for det in detections
            if str(det.get("category")) == "1"
        ]

        confidences = [float(det.get("conf", 0.0)) for det in animal_detections]
        area_ratios = [detection_area_ratio(det) for det in animal_detections]

        rows.append({
            "image_path": str(image_path),
            "megadetector_failure": failure,
            "md_num_animal_detections": len(animal_detections),
            "md_max_animal_confidence": max(confidences) if confidences else 0.0,
            "md_largest_bbox_area_ratio": max(area_ratios) if area_ratios else 0.0,
            "md_has_animal_020": any(c >= 0.20 for c in confidences),
            "md_has_animal_050": any(c >= 0.50 for c in confidences),
            "md_has_animal_080": any(c >= 0.80 for c in confidences),
        })

    return pd.DataFrame(rows)


if MD_JSON_PATH.exists():
    md_summary_df = parse_megadetector_json(MD_JSON_PATH)
    md_summary_df.to_csv(MD_SUMMARY_PATH, index=False)
    print(f"Parsed MegaDetector output: {MD_JSON_PATH}")
    print(f"Saved summary: {MD_SUMMARY_PATH}")
    display(md_summary_df.head())
else:
    md_summary_df = None
    print(f"MegaDetector JSON not found yet: {MD_JSON_PATH}")

## 9. Classifier errors vs MegaDetector detections

In [ ]:
if md_summary_df is not None:
    md_analysis_df = analysis_df.merge(
        md_summary_df,
        on="image_path",
        how="left",
    )

    for col in ["md_has_animal_020", "md_has_animal_050", "md_has_animal_080"]:
        md_analysis_df[col] = md_analysis_df[col].fillna(False).astype(bool)

    display(md_analysis_df[[
        "image_id",
        "true_label",
        "pred_label",
        "confidence",
        "lighting_label",
        "md_num_animal_detections",
        "md_max_animal_confidence",
        "md_largest_bbox_area_ratio",
        "md_has_animal_020",
        "md_has_animal_050",
    ]].head())
else:
    md_analysis_df = None
    print("Skipping detector analysis until MegaDetector results are available.")

In [ ]:
if md_analysis_df is not None:
    detector_threshold_col = "md_has_animal_020"
    md_analysis_df["true_has_target_animal"] = md_analysis_df["true_label"] != "empty"

    detector_table = pd.crosstab(
        md_analysis_df["true_has_target_animal"],
        md_analysis_df[detector_threshold_col],
        rownames=["true_non_empty"],
        colnames=["megadetector_has_animal"],
    )
    display(detector_table)

    detector_by_lighting = (
        md_analysis_df.groupby("lighting_label")
        .apply(lambda g: pd.Series({
            "count": len(g),
            "md_detect_rate_020": g["md_has_animal_020"].mean(),
            "md_detect_rate_050": g["md_has_animal_050"].mean(),
            "mean_md_conf": g["md_max_animal_confidence"].mean(),
        }))
        .reset_index()
    )
    display(detector_by_lighting)
else:
    print("No MegaDetector dataframe available.")

In [ ]:
if md_analysis_df is not None:
    model_pred_empty_detector_finds_animal = md_analysis_df[
        (md_analysis_df["pred_label"] == "empty")
        & (md_analysis_df["md_has_animal_020"])
    ]

    model_pred_animal_detector_finds_none = md_analysis_df[
        (md_analysis_df["pred_label"] != "empty")
        & (~md_analysis_df["md_has_animal_020"])
    ]

    empty_label_detector_finds_animal = md_analysis_df[
        (md_analysis_df["true_label"] == "empty")
        & (md_analysis_df["md_has_animal_020"])
    ]

    animal_label_detector_finds_none = md_analysis_df[
        (md_analysis_df["true_label"] != "empty")
        & (~md_analysis_df["md_has_animal_020"])
    ]

    summary = pd.DataFrame([
        {"case": "model_pred_empty_detector_finds_animal", "count": len(model_pred_empty_detector_finds_animal)},
        {"case": "model_pred_animal_detector_finds_none", "count": len(model_pred_animal_detector_finds_none)},
        {"case": "empty_label_detector_finds_animal", "count": len(empty_label_detector_finds_animal)},
        {"case": "animal_label_detector_finds_none", "count": len(animal_label_detector_finds_none)},
    ])
    display(summary)
else:
    print("No MegaDetector dataframe available.")

## 10. Visualize detector boxes for suspicious cases

In [ ]:
def get_md_image_record(json_path, image_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    image_path = str(image_path)
    for image_result in data.get("images", []):
        candidate = image_result.get("file") or image_result.get("filename")
        if str(candidate) == image_path:
            return image_result
    return None


def draw_md_boxes(image_path, detections, conf_threshold=0.20):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size

    for det in detections:
        if str(det.get("category")) != "1":
            continue
        conf = float(det.get("conf", 0.0))
        if conf < conf_threshold:
            continue
        x, y, w, h = det["bbox"]
        box = [x * width, y * height, (x + w) * width, (y + h) * height]
        draw.rectangle(box, outline="red", width=4)
        draw.text((box[0], max(0, box[1] - 14)), f"animal {conf:.2f}", fill="red")

    return image


def show_md_box_examples(df, title, n=8, ncols=4, conf_threshold=0.20):
    if md_analysis_df is None or not MD_JSON_PATH.exists():
        print("MegaDetector results unavailable.")
        return None

    examples = df.head(n).copy()
    if examples.empty:
        print("No examples to show.")
        return None

    nrows = int(np.ceil(len(examples) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.2 * ncols, 4.8 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, examples.iterrows()):
        record = get_md_image_record(MD_JSON_PATH, row["image_path"])
        detections = [] if record is None else record.get("detections", [])
        boxed_image = draw_md_boxes(row["image_path"], detections, conf_threshold=conf_threshold)
        ax.imshow(boxed_image)
        ax.set_title(
            f"T: {row['true_label']} | P: {row['pred_label']}\n"
            f"Cls conf: {row['confidence']:.2f} | MD: {row['md_max_animal_confidence']:.2f}",
            fontsize=9,
        )
        ax.axis("off")

    for ax in axes[len(examples):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()
    return fig


if md_analysis_df is not None:
    show_md_box_examples(
        empty_label_detector_finds_animal.sort_values("md_max_animal_confidence", ascending=False),
        title="Empty label but MegaDetector finds animal",
        n=8,
    )

    show_md_box_examples(
        animal_label_detector_finds_none.sort_values("confidence", ascending=False),
        title="Animal label but MegaDetector finds no animal",
        n=8,
    )
else:
    print("Skipping box visualizations until MegaDetector results are available.")

## 11. Controlled higher-resolution experiment — optional

This is a separate experiment. It does not use MegaDetector crops or detector outputs.

Run only if there is enough time. The goal is to compare the same transfer-learning idea at a larger input resolution.

In [ ]:
RUN_HIGH_RES_EXPERIMENT = False
HIGH_RES_IMG_SIZE = (320, 320)
HIGH_RES_EPOCHS_FROZEN = 6
HIGH_RES_EPOCHS_FINETUNE = 6
HIGH_RES_MODEL_PATH = MODELS_DIR / "mobilenetv2_finetuned_320_1000_per_class.keras"
HIGH_RES_RESULTS_PATH = REPORTS_DIR / "05_high_resolution_mobilenetv2_320_results.csv"

print("RUN_HIGH_RES_EXPERIMENT:", RUN_HIGH_RES_EXPERIMENT)
print("High-resolution image size:", HIGH_RES_IMG_SIZE)

In [ ]:
def make_tf_dataset(df, image_size, batch_size=BATCH_SIZE, shuffle=False):
    image_paths = df["image_path"].astype(str).values
    labels = df["label_idx"].astype("int32").values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)

    def load_tf_image(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, image_size)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = ds.map(load_tf_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def build_mobilenetv2_transfer_model(input_shape, num_classes):
    from tensorflow.keras.applications import MobileNetV2

    inputs = keras.Input(shape=input_shape)
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomZoom(0.10)(x)
    x = layers.RandomContrast(0.15)(x)
    x = layers.Rescaling(2.0, offset=-1.0, name="mobilenetv2_preprocessing")(x)

    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="mobilenetv2_320")


if RUN_HIGH_RES_EXPERIMENT:
    train_ds_320 = make_tf_dataset(train_df, HIGH_RES_IMG_SIZE, shuffle=True)
    val_ds_320 = make_tf_dataset(val_df, HIGH_RES_IMG_SIZE, shuffle=False)

    model_320 = build_mobilenetv2_transfer_model(
        input_shape=HIGH_RES_IMG_SIZE + (3,),
        num_classes=num_classes,
    )

    model_320.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]

    history_320_frozen = model_320.fit(
        train_ds_320,
        validation_data=val_ds_320,
        epochs=HIGH_RES_EPOCHS_FROZEN,
        callbacks=callbacks,
    )

    base_model_320 = model_320.get_layer("mobilenetv2_1.00_320")
    base_model_320.trainable = True
    fine_tune_at = 120
    for layer in base_model_320.layers[:fine_tune_at]:
        layer.trainable = False

    model_320.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    history_320_finetune = model_320.fit(
        train_ds_320,
        validation_data=val_ds_320,
        epochs=HIGH_RES_EPOCHS_FINETUNE,
        callbacks=callbacks,
    )

    y_true_320 = []
    y_prob_320 = []
    for images, labels in val_ds_320:
        probs = model_320.predict(images, verbose=0)
        y_true_320.extend(labels.numpy())
        y_prob_320.extend(probs)

    y_true_320 = np.array(y_true_320)
    y_prob_320 = np.array(y_prob_320)
    y_pred_320 = np.argmax(y_prob_320, axis=1)

    acc_320 = accuracy_score(y_true_320, y_pred_320)
    f1_320 = f1_score(y_true_320, y_pred_320, average="macro")

    high_res_results = pd.DataFrame([
        {"experiment": "mobilenetv2_224_existing", "val_accuracy": val_accuracy, "val_macro_f1": val_macro_f1},
        {"experiment": "mobilenetv2_320_controlled", "val_accuracy": acc_320, "val_macro_f1": f1_320},
    ])

    model_320.save(HIGH_RES_MODEL_PATH)
    high_res_results.to_csv(HIGH_RES_RESULTS_PATH, index=False)

    display(high_res_results)
else:
    print("High-resolution experiment disabled. Set RUN_HIGH_RES_EXPERIMENT = True to run it.")

## 12. Results summary

Complete after running the notebook.

In [ ]:
summary_rows = [
    {
        "section": "best_model_validation",
        "result": f"accuracy={val_accuracy:.4f}, macro_f1={val_macro_f1:.4f}",
    },
    {
        "section": "lighting_groups",
        "result": "; ".join(
            f"{row.lighting_label}: n={int(row.count)}, acc={row.accuracy:.3f}"
            for row in lighting_summary.itertuples()
        ),
    },
]

if md_analysis_df is not None:
    summary_rows.append({
        "section": "megadetector_available",
        "result": f"parsed {len(md_analysis_df)} validation rows with detector features",
    })
else:
    summary_rows.append({
        "section": "megadetector_available",
        "result": "not available yet; run MegaDetector and rerun sections 8-10",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(REPORTS_DIR / "05_summary.csv", index=False)
display(summary_df)

### Interpretation notes to complete after running

- Fine-tuned MobileNetV2 validation result: `...`
- Lowest-performing lighting group: `...`
- Main low-light error pattern: `...`
- Raccoon false-positive behavior by lighting: `...`
- MegaDetector agreement with empty/non-empty labels: `...`
- Detector cases worth manual audit: `...`
- Higher-resolution result, if run: `...`